# 4주차. 나나에게 손과 발을 달아주다

**부제:** SQLite로 대화 목록과 메시지 관리하기

## 0. 목표

이번 주에는 별도 Python 문제 파일 없이 노트북 안에서 SQLite 대화 저장 흐름을 직접 확인한다.

성취 기준:

- `conversations`와 `messages` table의 역할을 구분할 수 있다.
- `conversation_id`로 대화 목록과 메시지 로그를 연결할 수 있다.
- 보관 처리된 대화가 active 목록에서 빠지고 전체 목록에는 남는지 확인할 수 있다.

## 1. 준비

로컬 `Jupyter Notebook` 또는 `JupyterLab`에서 repo 루트를 기준으로 실행한다.

4주차는 OpenAI API를 호출하지 않는다. SQLite 파일은 기본적으로 `tmp/week04_conversations.sqlite3`에 만들어지고, 필요하면 `KANAMATE_WEEK04_DB_PATH`로 바꿀 수 있다.

처음 읽는 순서: `docs/orientation.md` → 이 노트북 → SQLite schema 셀 → 저장/조회/보관 실습 순서로 진행한다.

In [ ]:
from pathlib import Path
import json
import os
import sqlite3
import uuid
from datetime import datetime, timezone

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
DB_PATH = Path(os.getenv("KANAMATE_WEEK04_DB_PATH") or REPO_ROOT / "tmp" / "week04_conversations.sqlite3").resolve()
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

def show_json(value):
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))

def connect_db():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn

def initialize_conversation_db(reset: bool = False) -> Path:
    with connect_db() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS conversations (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                conversation_id TEXT UNIQUE NOT NULL,
                title TEXT NOT NULL,
                status TEXT NOT NULL DEFAULT 'active',
                created_at TEXT NOT NULL,
                updated_at TEXT NOT NULL,
                archived_at TEXT
            )
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS messages (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                message_id TEXT UNIQUE NOT NULL,
                conversation_id TEXT NOT NULL,
                role TEXT NOT NULL,
                content TEXT NOT NULL,
                created_at TEXT NOT NULL,
                FOREIGN KEY (conversation_id) REFERENCES conversations(conversation_id) ON DELETE CASCADE,
                CHECK (role IN ('system', 'user', 'assistant'))
            )
        """)
        if reset:
            conn.execute("DELETE FROM messages")
            conn.execute("DELETE FROM conversations")
        conn.commit()
    return DB_PATH

show_json({"db_path": str(initialize_conversation_db(reset=True))})

## 2. 개념

오늘의 큰 그림: 앱의 채팅 화면은 새로고침하면 사라지는 화면 상태가 아니라, 다시 조회할 수 있는 저장소 상태가 되어야 한다. SQLite는 작은 로컬 파일 하나로 대화 목록과 메시지 로그를 연습하기 좋은 저장소다.

반드시 이해할 것:

- `conversations`는 대화 하나의 제목, 상태, 생성/수정 시간을 저장한다.
- `messages`는 각 대화에 속한 user/assistant 메시지를 시간순으로 저장한다.
- `conversation_id`가 두 table을 연결한다.
- 보관은 삭제가 아니라 `status="archived"`로 상태를 바꾸는 일이다.

지금은 몰라도 되는 것:

- SQLite query optimizer
- 대규모 DB migration 전략
- 동시 접속과 lock tuning

막혔을 때 볼 값: `conversation_id`, `message_id`, `message_count`, `last_message`, `status`.

## 3. 기본 개념 실습

가장 작은 흐름은 "대화 생성 → 사용자 메시지 저장 → assistant 메시지 저장"이다. 먼저 한 대화 안에 두 메시지가 같은 `conversation_id`로 묶이는지 확인한다.

In [ ]:
VALID_ROLES = {"system", "user", "assistant"}

def row_to_dict(row):
    return dict(row)

def create_conversation(title: str) -> dict:
    created_at = now_iso()
    conversation_id = f"conv-{uuid.uuid4().hex[:12]}"
    with connect_db() as conn:
        conn.execute(
            """INSERT INTO conversations
               (conversation_id, title, status, created_at, updated_at, archived_at)
               VALUES (?, ?, ?, ?, ?, ?)""",
            (conversation_id, title.strip() or "새 대화", "active", created_at, created_at, None),
        )
        conn.commit()
        row = conn.execute("SELECT * FROM conversations WHERE conversation_id = ?", (conversation_id,)).fetchone()
    return row_to_dict(row)

def append_message(conversation_id: str, role: str, content: str) -> dict:
    if role not in VALID_ROLES:
        raise ValueError(f"지원하지 않는 role입니다: {role}")
    created_at = now_iso()
    message_id = f"msg-{uuid.uuid4().hex[:12]}"
    with connect_db() as conn:
        conn.execute(
            """INSERT INTO messages
               (message_id, conversation_id, role, content, created_at)
               VALUES (?, ?, ?, ?, ?)""",
            (message_id, conversation_id, role, content.strip(), created_at),
        )
        conn.execute("UPDATE conversations SET updated_at = ? WHERE conversation_id = ?", (created_at, conversation_id))
        conn.commit()
        row = conn.execute("SELECT * FROM messages WHERE message_id = ?", (message_id,)).fetchone()
    return row_to_dict(row)

conversation = create_conversation("발표 준비 대화")
user_message = append_message(conversation["conversation_id"], "user", "내일 발표 리허설에서 확인할 내용을 정리해줘")
assistant_message = append_message(conversation["conversation_id"], "assistant", "발표 자료, 데모 흐름, 예상 질문을 확인 목록으로 저장해둘게요.")

show_json({"conversation": conversation, "messages": [user_message, assistant_message]})

## 4. 카나메이트 확장 예제

대화가 여러 개가 되면 목록 화면과 상세 화면이 분리된다. 목록에서는 `message_count`와 `last_message`만 빠르게 보고, 상세에서는 전체 메시지 로그를 읽는다.

In [ ]:
def list_conversations(include_archived: bool = False) -> list[dict]:
    where = "" if include_archived else "WHERE c.status != 'archived'"
    with connect_db() as conn:
        rows = conn.execute(f"""
            SELECT c.conversation_id, c.title, c.status, c.created_at, c.updated_at, c.archived_at,
                   COUNT(m.id) AS message_count,
                   (SELECT content FROM messages last_m
                    WHERE last_m.conversation_id = c.conversation_id
                    ORDER BY last_m.id DESC LIMIT 1) AS last_message
            FROM conversations c
            LEFT JOIN messages m ON m.conversation_id = c.conversation_id
            {where}
            GROUP BY c.id
            ORDER BY c.updated_at DESC
        """).fetchall()
    return [row_to_dict(row) for row in rows]

def load_conversation(conversation_id: str) -> dict:
    with connect_db() as conn:
        conversation_row = conn.execute("SELECT * FROM conversations WHERE conversation_id = ?", (conversation_id,)).fetchone()
        message_rows = conn.execute("SELECT * FROM messages WHERE conversation_id = ? ORDER BY id", (conversation_id,)).fetchall()
    return {"conversation": row_to_dict(conversation_row), "messages": [row_to_dict(row) for row in message_rows]}

show_json({
    "active_list": list_conversations(),
    "conversation_detail": load_conversation(conversation["conversation_id"]),
})

## 5. 확장 예제 실행

보관 처리는 메시지를 지우지 않는다. active 목록에서는 사라지고, `include_archived=True` 전체 목록에는 남아 있어야 한다.

In [ ]:
def archive_conversation(conversation_id: str) -> dict:
    archived_at = now_iso()
    with connect_db() as conn:
        conn.execute(
            "UPDATE conversations SET status = 'archived', archived_at = ?, updated_at = ? WHERE conversation_id = ?",
            (archived_at, archived_at, conversation_id),
        )
        conn.commit()
        row = conn.execute("SELECT * FROM conversations WHERE conversation_id = ?", (conversation_id,)).fetchone()
    return row_to_dict(row)

archived = archive_conversation(conversation["conversation_id"])

show_json({
    "archived": archived,
    "active_after_archive": list_conversations(),
    "all_after_archive": list_conversations(include_archived=True),
})

## 6. 회고

확인 질문:

1. 대화 목록 table과 메시지 table을 나누는 이유는 무엇인가?
2. `conversation_id`는 어떤 row들을 연결하는가?
3. 보관과 삭제는 사용자 경험과 검증 방식에서 어떻게 다른가?
4. `message_count`와 `last_message`는 어떤 UI를 만들 때 유용한가?

작은 응용 과제: 대화 두 개를 만들고 하나만 보관한 뒤, active 목록과 전체 목록이 어떻게 다른지 비교한다.